Realización del EDA en referente al dataset bank-full.csv dataset:

Explicación de las variables de dicho dataset:

## 📊 Diccionario de Datos (Data Dictionary)

| Variable | Rol | Tipo de Dato | Tipo Democrático / Descripción | Unidades | ¿Valores Faltantes? |
| :--- | :--- | :--- | :--- | :--- | :---: |
| **`age`** | Feature | Integer | Edad del cliente | Años | No |
| **`job`** | Feature | Categorical | Tipo de trabajo (`admin.`, `blue-collar`, `entrepreneur`, `housemaid`, `management`, `retired`, `self-employed`, `services`, `student`, `technician`, `unemployed`, `unknown`) | — | No |
| **`marital`** | Feature | Categorical | Estado civil (`divorced`, `married`, `single`, `unknown`; nota: *divorced* incluye viudos) | — | No |
| **`education`** | Feature | Categorical | Nivel educativo (`basic.4y`, `basic.6y`, `basic.9y`, `high.school`, `illiterate`, `professional.course`, `university.degree`, `unknown`) | — | No |
| **`default`** | Feature | Binary | ¿Tiene crédito en mora/incumplimiento? (`yes`, `no`, `unknown`) | — | No |
| **`balance`** | Feature | Integer | Saldo medio anual | Euros (€) | No |
| **`housing`** | Feature | Binary | ¿Tiene préstamo hipotecario? (`yes`, `no`, `unknown`) | — | No |
| **`loan`** | Feature | Binary | ¿Tiene préstamo personal? (`yes`, `no`, `unknown`) | — | No |
| **`contact`** | Feature | Categorical | Tipo de comunicación de contacto (`cellular`, `telephone`) | — | **Sí** |
| **`day_of_week`** | Feature | Date / Categorical | Último día de la semana en que fue contactado | Días | No |
| **`month`** | Feature | Date / Categorical | Último mes del año en que fue contactado (`jan`, `feb`, ..., `nov`, `dec`) | Meses | No |
| **`duration`** | Feature | Integer | Duración del último contacto. | Segundos | No |
| **`campaign`** | Feature | Integer | Número de contactos realizados durante esta campaña para este cliente (incluye el último) | Contactos | No |
| **`pdays`** | Feature | Integer | Días transcurridos desde que el cliente fue contactado en una campaña previa (`-1` o `999` indica no contactado previamente) | Días | **Sí** |
| **`previous`** | Feature | Integer | Número de contactos realizados antes de esta campaña para este cliente | Contactos | No |
| **`poutcome`** | Feature | Categorical | Resultado de la campaña de marketing anterior (`failure`, `nonexistent`, `success`) | — | **Sí** |
| **`y`** | **Target** | Binary | **Variable objetivo:** ¿El cliente suscribió un depósito a plazo fijo? (`yes`, `no`) | — | No |

In [2]:
import pandas as pd  #for data manipulation operations
import numpy as np  #for numeric operations on data
import seaborn as sns  #for data visualization operations
import matplotlib.pyplot as plt  #for data visualization operations
from sklearn.preprocessing import LabelEncoder # for encoding
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler #for standardization
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import scipy.stats as st
from termcolor import colored

#from markupsafe import escape
#!pip install pandas-profiling
#import pandas_profiling

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score, roc_curve

from sklearn import model_selection
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier
#!pip install lightgbm
from lightgbm import LGBMClassifier

#ignore warnings
import warnings
warnings.filterwarnings("ignore")

#see model parametres
from sklearn import set_config
set_config(print_changed_only = False)

print(colored("\n THE REQUIRED LIBRARIES WERE SUCCESFULLY IMPORTED...", "green"))


 THE REQUIRED LIBRARIES WERE SUCCESFULLY IMPORTED...


In [3]:
# Cargamos el dataset

data = pd.read_csv('data/bank-full.csv', sep=';')

data.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [4]:
data['balance'].max()

np.int64(102127)

In [5]:
# Exploramos si existen nulos

data.isnull().sum()

age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

In [6]:
unknown_counts = (data == 'unknown').sum()
unknown_percentage = (data == 'unknown').mean() * 100

In [7]:
unknown_counts

age              0
job            288
marital          0
education     1857
default          0
balance          0
housing          0
loan             0
contact      13020
day              0
month            0
duration         0
campaign         0
pdays            0
previous         0
poutcome     36959
y                0
dtype: int64

In [8]:
unknown_percentage

age           0.000000
job           0.637013
marital       0.000000
education     4.107407
default       0.000000
balance       0.000000
housing       0.000000
loan          0.000000
contact      28.798301
day           0.000000
month         0.000000
duration      0.000000
campaign      0.000000
pdays         0.000000
previous      0.000000
poutcome     81.747805
y             0.000000
dtype: float64

El dataset no contiene valores missing explícitos (NaN), pero algunas variables contienen valores especiales que representan ausencia o desconocimiento de información, como pdays con -1 (que signicia que no se contactó, información util) y poutcome con "unknown". Este última podríamos tratarlo como NaN, pero en prinicipio para realizar el EDA lo mantendremos como una categoría más con el fin de ver si tiene alguna relación sustancial conocida con la variable y que nuestro modelo le venga bien entender.

In [9]:
# Comprobamos que no existan duplicados

data.duplicated().sum()

np.int64(0)

In [10]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        45211 non-null  int64
 1   job        45211 non-null  str  
 2   marital    45211 non-null  str  
 3   education  45211 non-null  str  
 4   default    45211 non-null  str  
 5   balance    45211 non-null  int64
 6   housing    45211 non-null  str  
 7   loan       45211 non-null  str  
 8   contact    45211 non-null  str  
 9   day        45211 non-null  int64
 10  month      45211 non-null  str  
 11  duration   45211 non-null  int64
 12  campaign   45211 non-null  int64
 13  pdays      45211 non-null  int64
 14  previous   45211 non-null  int64
 15  poutcome   45211 non-null  str  
 16  y          45211 non-null  str  
dtypes: int64(7), str(10)
memory usage: 5.9 MB


In [11]:
# Revisamos los types de cada variable para luego preparar los datos

categorical_columns = data.select_dtypes(include="str").columns

data[categorical_columns] = data[categorical_columns].astype("object")

data.info()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


In [12]:
data.nunique()

age            77
job            12
marital         3
education       4
default         2
balance      7168
housing         2
loan            2
contact         3
day            31
month          12
duration     1573
campaign       48
pdays         559
previous       41
poutcome        4
y               2
dtype: int64

| Variable       | Nº de categorías |
| -------------- | ---------------: |
| `job`          |               12 |
| `marital`      |                3 |
| `education`    |                4 |
| `default`      |                2 |
| `housing`      |                2 |
| `loan`         |                2 |
| `contact`      |                3 |
| `month`        |               12 |
| `poutcome`     |                4 |
| `y` *(target)* |                2 |


In [13]:
data.describe().T.style.background_gradient(cmap = "magma")

,count,mean,std,min,25%,50%,75%,max
age,45211.000000,40.936210,10.618762,18.000000,33.000000,39.000000,48.000000,95.000000
balance,45211.000000,1362.272058,3044.765829,-8019.000000,72.000000,448.000000,1428.000000,102127.000000
day,45211.000000,15.806419,8.322476,1.000000,8.000000,16.000000,21.000000,31.000000
duration,45211.000000,258.163080,257.527812,0.000000,103.000000,180.000000,319.000000,4918.000000
campaign,45211.000000,2.763841,3.098021,1.000000,1.000000,2.000000,3.000000,63.000000
pdays,45211.000000,40.197828,100.128746,-1.000000,-1.000000,-1.000000,-1.000000,871.000000
previous,45211.000000,0.580323,2.303441,0.000000,0.000000,0.000000,0.000000,275.000000


TARGET

In [14]:
porcentaje_obj = (data['y'].value_counts(normalize=True))
porcentaje_obj

y
no     0.883015
yes    0.116985
Name: proportion, dtype: float64

Vemos que claramente hay una predominancia de la clase "no". Al ser desbalanceado no nos fijaremos solamente en accuracy al evaluar, porque un modelo que solo prediga "no" sería de un 88% de accuracy. Debemos también tener en cuenta esto para más tarde realizar un "stratify" adecuado"

VARIABLES CATEGÓRICAS

1. Job

In [ ]:
job_split = data['job'].value_counts(normalize=True)*100

job
blue-collar      21.525735
management       20.919688
technician       16.803433
admin.           11.437482
services          9.188029
retired           5.007631
self-employed     3.492513
entrepreneur      3.289023
unemployed        2.882042
housemaid         2.742695
student           2.074716
unknown           0.637013
Name: proportion, dtype: float64

In [50]:
pd.crosstab(data["job"], data["y"])

y,no,yes
job,,
admin.,4540,631
blue-collar,9024,708
entrepreneur,1364,123
housemaid,1131,109
management,8157,1301
retired,1748,516
self-employed,1392,187
services,3785,369
student,669,269


In [48]:
job_split_obj = data[['job', 'y']].groupby(['job']).value_counts(normalize=True).unstack(fill_value=0)

print("Categoría con mayor YES:", job_split_obj["yes"].idxmax())
print("Porcentaje:", job_split_obj["yes"].max())

print("Categoría con mayor NO:", job_split_obj["no"].idxmax())
print("Porcentaje:", job_split_obj["no"].max())

print('Media de aceptación de las categorías: ', job_split_obj['yes'].mean())

Categoría con mayor YES: student
Porcentaje: 0.2867803837953092
Categoría con mayor NO: blue-collar
Porcentaje: 0.9272503082614056
Media de aceptación de las categorías:  0.13404661488833422


In [47]:
job_split_obj

y,no,yes
job,,
admin.,0.877973,0.122027
blue-collar,0.927250,0.072750
entrepreneur,0.917283,0.082717
housemaid,0.912097,0.087903
management,0.862444,0.137556
retired,0.772085,0.227915
self-employed,0.881571,0.118429
services,0.911170,0.088830
student,0.713220,0.286780


Anotaciones:

--> Blue-Collar con mayor porcentaje de no, aunque categoría más llamada


--> Management estima un porcentaje similar de gente total, duplicando la tasa de sis (Una posible explicación podría estar relacionada con diferencias en la situación financiera de ambos grupos, como préstamos, balance o vivienda, que deberían analizarse posteriormente)


--> Estudiantes y retirados con mayor porcentaje de si, aunque representan solo el 2% y 5% correspondiente. Los grupos student y retired presentan las mayores tasas de contratación (28.68% y 22.79%, respectivamente). Esto podría estar relacionado con diferencias en edad, situación económica, balance u otras características, por lo que sería necesario analizar estas variables conjuntamente antes de establecer una explicación.


--> Los demás en una horquilla bastante similar de 7% a 14%

2. Marital

In [56]:
marital_split = data['marital'].value_counts(normalize=True)*100

marital_split

marital
married     60.193316
single      28.289576
divorced    11.517109
Name: proportion, dtype: float64

In [57]:
pd.crosstab(data["marital"], data["y"])

y,no,yes
marital,,
divorced,4585,622
married,24459,2755
single,10878,1912


In [58]:
marital_split_obj = data[['marital', 'y']].groupby(['marital']).value_counts(normalize=True).unstack(fill_value=0)

print("Categoría con mayor YES:", marital_split_obj["yes"].idxmax())
print("Porcentaje:", marital_split_obj["yes"].max())

print("Categoría con mayor NO:", marital_split_obj["no"].idxmax())
print("Porcentaje:",  marital_split_obj["no"].max())

print('Media de aceptación de las categorías: ', marital_split_obj['yes'].mean())

Categoría con mayor YES: single
Porcentaje: 0.1494917904612979
Categoría con mayor NO: married
Porcentaje: 0.8987653413684134
Media de aceptación de las categorías:  0.12339367648848665


In [59]:
marital_split_obj

y,no,yes
marital,,
divorced,0.880545,0.119455
married,0.898765,0.101235
single,0.850508,0.149492


A simple vista no parece que influya mucho el estado civil a la hora de acceder o no al deposito economico. Todos aparecen con la misma tasa de porcentaje en yes, despuntando un poco la clase "single" (quizá por la comodida de decidir solo?)

In [63]:
categorical_cols = data.select_dtypes(include="object").columns.drop("y")

for col in categorical_cols:

    # % que representa cada categoría en el dataset
    distribution = (
        data[col]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
        .rename("dataset_%")
    )

    # Cantidad de NO y YES
    counts = pd.crosstab(data[col], data["y"])

    # % de NO y YES dentro de cada categoría
    percentages = (
        pd.crosstab(data[col], data["y"], normalize="index")
        .mul(100)
        .round(2)
    )

    # Construimos la tabla final
    result = pd.DataFrame({
        "dataset_%": distribution,
        "no_count": counts["no"],
        "no_%": percentages["no"],
        "yes_count": counts["yes"],
        "yes_%": percentages["yes"]
    })

    print("\n" + "=" * 70)
    print(f"VARIABLE: {col}")
    print("=" * 70)

    print(result)

    # Resumen
    print(
        f"\nMayor YES: {result['yes_%'].idxmax()} "
        f"({result['yes_%'].max():.2f}%)"
    )

    print(
        f"Mayor NO: {result['no_%'].idxmax()} "
        f"({result['no_%'].max():.2f}%)"
    )

    print(
        f"Media YES: {result['yes_%'].mean():.2f}%"
    )


VARIABLE: job
               dataset_%  no_count   no_%  yes_count  yes_%
job                                                        
admin.             11.44      4540  87.80        631  12.20
blue-collar        21.53      9024  92.73        708   7.27
entrepreneur        3.29      1364  91.73        123   8.27
housemaid           2.74      1131  91.21        109   8.79
management         20.92      8157  86.24       1301  13.76
retired             5.01      1748  77.21        516  22.79
self-employed       3.49      1392  88.16        187  11.84
services            9.19      3785  91.12        369   8.88
student             2.07       669  71.32        269  28.68
technician         16.80      6757  88.94        840  11.06
unemployed          2.88      1101  84.50        202  15.50
unknown             0.64       254  88.19         34  11.81

Mayor YES: student (28.68%)
Mayor NO: blue-collar (92.73%)
Media YES: 13.40%

VARIABLE: marital
          dataset_%  no_count   no_%  yes_count 

Anotaciones:

- Education:

       --> Predominance de clase en eduación "secundaria",
       --> Clase con mayor si --> terciaria
       --> Clase con menor si --> primaria (quizá fiarse menos al tener una menos base de conomía?)

- Default:

       --> Muchos con no (muchos sin incumplimiento de un credito)
       --> yes es una clase muy minoritaria, con una tasa de yes bastante baja (normal al tener incumplimiento con otros bancos), no conviene ir por ahí

- Housing:

      --> muchos con préstmao hipotecarios tiene una tasa de menos de la mitad con respecto a no tenerla, con porcentajes de dataset no muy dispares --> la gente con prestamos de casa no suele hacer depósito (ya sea porque no pueden con el banco original o menor capital)

- Loan:
      --> no tener loan implica un mayor porcentaje de yes (el doble con respecto a no tenerlo), aunque con mucho mas porcentaje de dataset total
      --> Los clientes que tienen un préstamo pueden disponer de una menor capacidad de ahorro o de una menor cantidad de dinero disponible para inmovilizar en un depósito a plazo, lo que podría contribuir a su menor tasa de contratación.